In [ ]:
# import knihoven
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from uncertainties import ufloat as uf
from scipy.optimize import curve_fit
from pathlib import Path
from uncertainties import correlated_values

# path config
downloads_dir = Path.home() / 'Downloads'
# ujistime se, ze slozka existuje (kdyby nahodou ne)
downloads_dir.mkdir(parents=True, exist_ok=True)
# output_path = downloads_dir / 'muj_graf.png'

In [ ]:
# definice funkcí

# nejistota typu A 
def unc_A(data):
    """
    Výpočet nejistoty typu A
    data : list nebo numpy array
         měření
    return : float
        nejistota typu A
    """
    data = np.array(data)
    n = len(data)

    s = np.std(data, ddof=1)
    u_A = s / np.sqrt(n)
    
    return u_A

# nacteni z df do jednotlivych arrays
def df_to_arrays(df, sloupce, od=0, do=None, dtype=float):
    if isinstance(sloupce, str):
        sloupce = [sloupce]

    chybejici = [s for s in sloupce if s not in df.columns]
    if chybejici:
        raise KeyError(f"Chybejici sloupce: {chybejici}")

    return [df[s].iloc[od:do].dropna().to_numpy(dtype=dtype) for s in sloupce]

# array do latex
def array_to_latex(*array, dig=1, format='u', style='L', only_return=False):
    output = [f'${u:.{dig}{format}{style}}$' for u in array]
    if only_return:
        return output
    else:
        for s in output:
            print(s)

# prepocet z popt a pcov na ufloaty
def popt_pcov_to_ufloat(popt, pcov, correlated=False):
    """
    Převede výsledek curve_fit (popt, pcov) na ufloaty.

    Parametry
    ---------
    popt : array_like   optimální parametry z curve_fit
    pcov : array_like   kovarianční matice z curve_fit
    correlated : bool   True  -> ufloaty nesou korelace mezi parametry
                        False -> nezávislé ufloaty (jen sqrt diagonály pcov)

    Vrací
    -----
    tuple[ufloat, ...]  parametry s nejistotami
    """
    popt = np.asarray(popt, dtype=float)
    pcov = np.asarray(pcov, dtype=float)

    if correlated:
        return tuple(correlated_values(popt, pcov))

    perr = np.sqrt(np.diag(pcov))
    return tuple(uf(v, e) for v, e in zip(popt, perr))

In [ ]:
# nacteni google tabulek
sheet_id = "1P7mSp8lbFBDA2KqiddyACMX5QHoSjMJJDc4O2W-uRmY"
gid = "0"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_bk7 = pd.read_csv(url)
df_bk7

In [ ]:
# tlusta vrstva
vln_delka, propustnost = df_to_arrays(df_bk7, ['vln delka', 'propustnost'])
propustnost = propustnost/100

# propustnost bk7
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(r'Propustnost', fontsize=20)
ax.set_xlabel(r'$\lambda$ [nm]', fontsize=20)
ax.plot(vln_delka, propustnost, label='Naměřené hodnoty')
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
plt.savefig(downloads_dir / 'bk7_propust.pdf', dpi=300, bbox_inches='tight')


index_lomu = (1 + (1 - propustnost**2)**0.5)/propustnost

vln_delka_linspace = np.linspace(400, 900, 1000)

cauchy_mask = (vln_delka >= 400) & (vln_delka <= 900)
vln_delka_cauchy = vln_delka[cauchy_mask]
index_lomu_cauchy = index_lomu[cauchy_mask]

def cauchy(delka_VLN, A, B):
    x_func = 1 /(delka_VLN**2)
    return A + B*x_func

p0 = [1.5, 10000]
popt, pcov = curve_fit(cauchy, vln_delka_cauchy, index_lomu_cauchy, p0=p0)
A, B = popt_pcov_to_ufloat(popt, pcov)
print(f'Cauchyho parametry skla BK7 jsou:\nA = {A:.1uP}\nB = {B:.1uP}')


fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(r'Index lomu $n$', fontsize=20)
ax.set_xlabel(r'$\lambda$ [nm]', fontsize=20)
ax.plot(vln_delka, index_lomu, label='Vypočítané hodnoty')
ax.plot(vln_delka_linspace, cauchy(vln_delka_linspace, *popt), color='red', linestyle='-',
    label=
    r'Fit Cauchyho vztahem $A + B/\lambda^2$:''\n'
    #r'$A + B/\lambda^2$''\n'
    fr'$A = {A:.1uL}$''\n'
    fr'$B = {B:.1uL}\,\mathrm{{nm^2}}$')

vln_delka_cauchy_mimo = np.concatenate([np.linspace(300, 400, 1000), [np.nan], np.linspace(900, 1000, 1000)])
ax.plot(vln_delka_cauchy_mimo, cauchy(vln_delka_cauchy_mimo, *popt), color='red', linestyle='--', label='Cauchyho vztah mimo\ninterval 400-900 nm')


axins = ax.inset_axes([0.15, 0.3, 0.55, 0.55])
axins.plot(vln_delka_cauchy, index_lomu_cauchy)
axins.plot(vln_delka_linspace, cauchy(vln_delka_linspace, *popt), color='red')
axins.set_xlim(400, 900)
ax.indicate_inset_zoom(
    axins,
    edgecolor="black",
    linewidth=1,
    alpha=0.75,
    linestyle='--'
)

ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
plt.savefig(downloads_dir / 'bk7.pdf', dpi=300, bbox_inches='tight')

In [ ]:
# nacteni google tabulek
sheet_id = "1P7mSp8lbFBDA2KqiddyACMX5QHoSjMJJDc4O2W-uRmY"
gid = "2142378828"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_subs_vrstv = pd.read_csv(url)
df_subs_vrstv

In [ ]:
# substrat a tenka vrstva

substrat, vrstva = df_to_arrays(df_subs_vrstv, ['substrat', 'vrstva'])
substrat, vrstva = substrat/100, vrstva/100

# fit substratu pro pocatecni parametry
index_lomu_subs = (1 + (1 - substrat**2)**0.5)/substrat
def cauchy(delka_VLN, A, B):
    x_func = 1 /(delka_VLN**2)
    return A + B*x_func
p0 = [1.54, 4045]
popt_subs, _ = curve_fit(cauchy, vln_delka_cauchy, index_lomu_subs[cauchy_mask], p0=p0)
A_subs, B_subs = popt_subs
print(f'Optimálni startovací parametry pro subsrát jsou:\n{popt_subs}')

# fit tenke vrstvy
def propust(lambd_nm, Av, Bv, d_nm, As, Bs):
    n_s = As + Bs/(lambd_nm**2)
    n_v = Av + Bv/(lambd_nm**2)
    beta = ((2*np.pi)/lambd_nm)*n_v*d_nm  # lambd i d v nm
    T_vs = (2*n_s)/(1 + n_s**2 + ((n_v**2 - 1)/(2))*(1 - (n_s**2)/(n_v**2))*((np.sin(beta))**2))
    return T_vs

p0 = [2.21, 63000, 525, 1.54, 4045]
popt, pcov = curve_fit(propust, vln_delka_cauchy, vrstva[cauchy_mask], p0=p0)
Av, Bv, d_nm, _, _ = popt
print(f'Parametry fitu jsou:\n{popt}')
unc_popt = np.sqrt(np.diag(pcov))
_, _, unc_d_nm, _, _ = unc_popt
array_to_latex(uf(d_nm, unc_d_nm))

# namerena data a fit
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(r'Propustnost', fontsize=20)
ax.set_xlabel(r'$\lambda$ [nm]', fontsize=20)
ax.plot(vln_delka, substrat, label='Naměřené hodnoty substrát')
ax.plot(vln_delka, vrstva, label='Naměřené hodnoty vrstva')
ax.plot(vln_delka_cauchy, propust(vln_delka_cauchy, *popt), label='Fit', color='red')
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
plt.savefig(downloads_dir / 'substrat_vrstva.pdf', dpi=300, bbox_inches='tight')

# index lomu vrstvy
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(r'Index lomu $n$', fontsize=20)
ax.set_xlabel(r'$\lambda$ [nm]', fontsize=20)
ax.plot(vln_delka_cauchy, cauchy(vln_delka_cauchy, Av, Bv), label='Fit')
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
plt.savefig(downloads_dir / 'index_vrstva.pdf', dpi=300, bbox_inches='tight')

In [ ]:
# nacteni google tabulek
sheet_id = "1P7mSp8lbFBDA2KqiddyACMX5QHoSjMJJDc4O2W-uRmY"
gid = "1962216730"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_lambert_beer = pd.read_csv(url)
df_lambert_beer

In [ ]:
# lambert beer
tloustka_dest = np.array([3.60, 3.55, 3.55, 3.60, 3.50])/1000
dest_d = np.mean(tloustka_dest)
dest_d_np = np.array([dest_d, 2*dest_d, 3*dest_d, 4*dest_d])
lamber_beer_list = ['dest1', 'dest2', 'dest3', 'dest4']
dest1, dest2, dest3, dest4 = df_to_arrays(df_lambert_beer, lamber_beer_list)
dest1, dest2, dest3, dest4 = dest1/100, dest2/100, dest3/100, dest4/100
lndest1, lndest2, lndest3, lndest4, = np.log(dest1), np.log(dest2), np.log(dest3), np.log(dest4)

fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(r'Propustnost', fontsize=20)
ax.set_xlabel(r'$\lambda$ [nm]', fontsize=20)
ax.plot(vln_delka, dest1)
ax.plot(vln_delka, dest2)
ax.plot(vln_delka, dest3)
ax.plot(vln_delka, dest4)
ax.axvline(vln_delka[200])
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
#ax.legend(fontsize=15)
plt.savefig(downloads_dir / 'lambert_beer.pdf', dpi=300, bbox_inches='tight')

def absorpce(d, alph):
    lnT = -alph*d
    return lnT
dest_d_np_list200 = [lndest1[200], lndest2[200], lndest3[200], lndest4[200]]

alpha, _ = curve_fit(absorpce, dest_d_np, dest_d_np_list200)
dest_d_np_linspace = np.linspace(0, 4*dest_d, 1000)

fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(r'$ln T$', fontsize=20)
ax.set_xlabel(r'$d$ [m]', fontsize=20)
ax.plot(dest_d_np_linspace, absorpce(dest_d_np_linspace, alpha))
ax.scatter(dest_d_np, dest_d_np_list200)

ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
#ax.legend(fontsize=15)
plt.savefig(downloads_dir / 'nplog.pdf', dpi=300, bbox_inches='tight')

In [ ]:
from scipy.optimize import curve_fit

T_all = np.vstack([dest1, dest2, dest3, dest4])   # tvar (4, N_lambda)
lnT_all = np.log(T_all)

idx_list = [100, 200, 350]   # zvolte vhodné vlnové délky

def lin(d, alph):
    return -alph * d

fig, axes = plt.subplots(1, len(idx_list), figsize=(18, 6))
d_fit = np.linspace(0, dest_d_np.max(), 100)

for ax, i in zip(axes, idx_list):
    y = lnT_all[:, i]
    popt, pcov = curve_fit(lin, dest_d_np, y)
    alpha, alpha_err = popt[0], np.sqrt(pcov[0, 0])

    # R^2
    res = y - lin(dest_d_np, alpha)
    r2 = 1 - np.sum(res**2) / np.sum((y - y.mean())**2)

    ax.scatter(dest_d_np*1e3, y, label='data')
    ax.plot(d_fit*1e3, lin(d_fit, alpha), label=fr'fit: $\alpha$ = {alpha/1e3:.3f} ± {alpha_err/1e3:.3f} mm$^{{-1}}$')
    ax.set_title(f'λ = {vln_delka[i]:.0f} nm, $R^2$ = {r2:.4f}')
    ax.set_xlabel('d [mm]')
    ax.set_ylabel('ln T')
    ax.grid(True, alpha=0.7)
    ax.legend()
    print(vln_delka[i], alpha, alpha_err, r2)

plt.savefig(downloads_dir / 'nplog_claude.pdf', dpi=300, bbox_inches='tight')

In [ ]:
# absorpční koeficient spektra -- 3 c)
dest_d_abs = dest_d * 100 # v cm
absorpcni_koef1 = (-1*lndest1)/dest_d_abs
absorpcni_koef2 = (-1*lndest2)/(2*dest_d_abs)
absorpcni_koef3 = (-1*lndest4)/(3*dest_d_abs)
absorpcni_koef4 = (-1*lndest3)/(4*dest_d_abs)

fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(r'$\alpha$ [cm$^{-1}$]', fontsize=20)
ax.set_xlabel(r'$\lambda$ [nm]', fontsize=20)
ax.plot(vln_delka, absorpcni_koef1, label='alpha1 - 1 destička')
ax.plot(vln_delka, absorpcni_koef2, label='alpha2 - 2 destičky')
ax.plot(vln_delka, absorpcni_koef3, label='alpha3 - 3 destičky')
ax.plot(vln_delka, absorpcni_koef4, label='alpha4 - 4 destičky')
#ax.axvline(390)
#ax.axvline(760)
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
plt.savefig(downloads_dir / 'nplog.pdf', dpi=300, bbox_inches='tight')